In [1]:
import os
os.chdir("../")
%pwd

'c:\\1_Code\\Class\\ChickenDisease_Haris'

In [2]:
# Update Config.yaml
# update Entity
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig: 
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list


@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path

In [3]:
# Update Configurations manager
import os
from cnnClassifier.constants import * 
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf


In [4]:
class ConfigurationManager: 
    def __init__(
            self, 
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH): 
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_prepare_callback_config(self) -> PrepareCallbacksConfig:
        config = self.config.prepare_callbacks
        model_ckpt_dir = os.path.dirname(config.checkpoint_model_filepath)
        create_directories([
            Path(model_ckpt_dir),
            Path(config.tensorboard_root_log_dir)
        ])

        prepare_callback_config = PrepareCallbacksConfig(
            root_dir=Path(config.root_dir),
            tensorboard_root_log_dir=Path(config.tensorboard_root_log_dir),
            checkpoint_model_filepath=Path(config.checkpoint_model_filepath)
        )

        return prepare_callback_config
    
    def get_training_config(self) -> TrainingConfig: 
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "Chicken-fecal-images")
        create_directories([Path(training.root_dir)])

        training_config = TrainingConfig(
            root_dir = Path(training.root_dir), 
            trained_model_path = Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data = Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE,
        )

        return training_config

In [5]:
# Update Components
import time

In [6]:
class PrepareCallback:
    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config


    
    @property
    def _create_tb_callbacks(self):
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_running_log_dir = os.path.join(
            self.config.tensorboard_root_log_dir,
            f"tb_logs_at_{timestamp}",
        )
        return tf.keras.callbacks.TensorBoard(log_dir=tb_running_log_dir)
    

    @property
    def _create_ckpt_callbacks(self):
        return tf.keras.callbacks.ModelCheckpoint(
            filepath=str(self.config.checkpoint_model_filepath),
            save_best_only=True
        )


    def get_tb_ckpt_callbacks(self):
        return [
            self._create_tb_callbacks,
            self._create_ckpt_callbacks
        ]

In [7]:
# Create new component

import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [8]:
class Training: 
    def __init__(self, config: TrainingConfig): 
        self.config = config
    
    def get_base_model(self): 
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )
        # Must create a NEW optimizer instance from the loaded one's config.
        # Reusing the same optimizer object causes "Unknown variable" error
        # because its internal state is tied to the original variable keys.
        old_opt = self.model.optimizer
        new_optimizer = old_opt.__class__.from_config(old_opt.get_config())
        self.model.compile(
            optimizer=new_optimizer,
            loss=self.model.loss,
            metrics=['accuracy'],
            run_eagerly=True
        )

    def train_valid_generator(self): 
        image_size = self.config.params_image_size[:-1]  # e.g., [224, 224]

        # Use pure TF ops (not Keras layers) inside map() to avoid
        # "numpy() is only available when eager execution is enabled" error
        def normalize(x, y):
            return tf.cast(x, tf.float32) / 255.0, y

        def augment_and_normalize(x, y):
            x = tf.cast(x, tf.float32) / 255.0
            x = tf.image.random_flip_left_right(x)
            x = tf.image.random_brightness(x, max_delta=0.2)
            x = tf.image.random_contrast(x, lower=0.8, upper=1.2)
            x = tf.clip_by_value(x, 0.0, 1.0)
            return x, y

        # Validation dataset (no augmentation, no shuffle)
        self.valid_generator = tf.keras.utils.image_dataset_from_directory(
            directory=self.config.training_data,
            validation_split=0.20,
            subset="validation",
            seed=42,
            image_size=image_size,
            batch_size=self.config.params_batch_size,
            label_mode='categorical',
            shuffle=False
        ).map(normalize, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

        if self.config.params_is_augmentation: 
            # Apply augmentation + normalization for training
            self.train_generator = tf.keras.utils.image_dataset_from_directory(
                directory=self.config.training_data,
                validation_split=0.20,
                subset="training",
                seed=42,
                image_size=image_size,
                batch_size=self.config.params_batch_size,
                label_mode='categorical',
                shuffle=True
            ).map(augment_and_normalize, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
        else: 
            # No augmentation — just normalize
            self.train_generator = tf.keras.utils.image_dataset_from_directory(
                directory=self.config.training_data,
                validation_split=0.20,
                subset="training",
                seed=42,
                image_size=image_size,
                batch_size=self.config.params_batch_size,
                label_mode='categorical',
                shuffle=True
            ).map(normalize, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model): 
        model.save(path)

    def train(self, callback_list: list): 
        self.history = self.model.fit(
            self.train_generator, 
            epochs=self.config.params_epochs,
            validation_data=self.valid_generator,
            callbacks=callback_list
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )


In [9]:
%pwd

'c:\\1_Code\\Class\\ChickenDisease_Haris'

In [10]:
os.chdir(r"C:\1_Code\Class\ChickenDisease_Haris")

In [11]:
%pwd

'C:\\1_Code\\Class\\ChickenDisease_Haris'

In [12]:
# Create pipeline
try: 
    config = ConfigurationManager()
    prepare_callbacks_config = config.get_prepare_callback_config()
    prepare_callbacks = PrepareCallback(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()

    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train(callback_list = callback_list)

except Exception as e:
    raise e

[2026-05-19 16:10:11,894: INFO: common] yaml file: config\config.yaml loaded successfully
[2026-05-19 16:10:11,901: INFO: common] yaml file: params.yaml loaded successfully
[2026-05-19 16:10:11,904: INFO: common] directory created at: artifacts
[2026-05-19 16:10:11,906: INFO: common] directory created at: artifacts\prepare_callbacks\checkpoint_dir
[2026-05-19 16:10:11,908: INFO: common] directory created at: artifacts\prepare_callbacks\tensorboard_log_dir
[2026-05-19 16:10:11,912: INFO: common] directory created at: artifacts\training
[2026-05-19 16:10:12,316: WARNING: saving_utils] Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.
Found 390 files belonging to 2 classes.
Using 78 files for validation.
Found 390 files belonging to 2 classes.
Using 312 files for training.
Epoch 1/2
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.4580 - loss: 15.6652[2026-05-19 16:12:39,972: WARNING: s